# Cleaning Twelfth Night

Turning the play into a table (one row per speech) and making a cleaned version of each speech for word frequency. Cleaning steps are from the week 6 notebook (6-2).

In [1]:
import re
import pandas as pd

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ps457\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Load the play

In [2]:
with open('Twelfth_night_full_text.txt', encoding='utf-8') as f:
    text = f.read()

# skip the title page and character list
text = text[text.index('ACT 1'):]
print(text[:500])

ACT 1
=====

Scene 1
[Enter Orsino, Duke of Illyria, Curio, and other Lords,
with Musicians playing.]


ORSINO
If music be the food of love, play on.
Give me excess of it, that, surfeiting,
The appetite may sicken and so die.
That strain again! It had a dying fall.
O, it came o'er my ear like the sweet sound
That breathes upon a bank of violets,
Stealing and giving odor. Enough; no more.
'Tis not so sweet now as it was before.
O spirit of love, how quick and fresh art thou,
That, notwith


## Split into acts and scenes

Acts and scenes are marked like `ACT 1` and `Scene 1` with a line of `=====` under them.

In [3]:
scenes = []

acts = re.split(r'ACT (\d+)\n=+', text)
for i in range(1, len(acts), 2):
    act = int(acts[i])
    scene_parts = re.split(r'Scene (\d+)\n=+', acts[i + 1])
    for j in range(1, len(scene_parts), 2):
        scenes.append((act, int(scene_parts[j]), scene_parts[j + 1]))

print(len(scenes), 'scenes')

18 scenes


## Stage directions

Stage directions are inside [brackets]. We save them in their own table (so we can count entrances and exits) and remove them from the speeches.

In [4]:
directions = []

for act, scene, scene_text in scenes:
    for d in re.findall(r'\[(.*?)\]', scene_text, re.DOTALL):
        d = ' '.join(d.split())
        if 'Enter' in d:
            kind = 'enter'
        elif 'exit' in d.lower():
            kind = 'exit'
        else:
            kind = 'other'
        directions.append({'act': act, 'scene': scene, 'kind': kind, 'direction': d})

directions_df = pd.DataFrame(directions)
directions_df.head()

,act,scene,kind,direction
0,1,1,enter,"Enter Orsino, Duke of Illyria, Curio, and othe..."
1,1,1,enter,Enter Valentine.
2,1,1,exit,They exit.
3,1,2,enter,"Enter Viola, a Captain, and Sailors."
4,1,2,other,giving him money


## Speeches

Speaker names are in capitals at the start of a line, like `ORSINO` or `CURIO  The hart.`, so a new speech starts every time we see one. Lines after that belong to the same speaker until the next name.

In [5]:
speaker_pattern = re.compile(r"^([A-Z][A-Z']+(?: [A-Z][A-Z']+)*)(?=,|\s{2,}|$)")

rows = []

for act, scene, scene_text in scenes:
    scene_text = re.sub(r'\[.*?\]', '', scene_text, flags=re.DOTALL)  # remove stage directions
    for line in scene_text.split('\n'):
        line = line.strip()
        if line == '':
            continue
        match = speaker_pattern.match(line)
        if match:
            rows.append({'act': act, 'scene': scene, 'speaker': match.group(1),
                         'text': line[match.end():].lstrip(' ,')})
        elif rows:
            rows[-1]['text'] += ' ' + line

speeches = pd.DataFrame(rows)
speeches['text'] = speeches['text'].apply(lambda t: ' '.join(t.split()))
speeches = speeches[speeches['text'] != ''].reset_index(drop=True)
speeches.head(10)

,act,scene,speaker,text
0,1,1,ORSINO,"If music be the food of love, play on. Give me..."
1,1,1,CURIO,"Will you go hunt, my lord?"
2,1,1,ORSINO,"What, Curio?"
3,1,1,CURIO,The hart.
4,1,1,ORSINO,"Why, so I do, the noblest that I have. O, when..."
5,1,1,VALENTINE,"So please my lord, I might not be admitted, Bu..."
6,1,1,ORSINO,"O, she that hath a heart of that fine frame To..."
7,1,2,VIOLA,"What country, friends, is this?"
8,1,2,CAPTAIN,"This is Illyria, lady."
9,1,2,VIOLA,And what should I do in Illyria? My brother he...


In [6]:
# checking: should be 5 acts, 18 scenes, and only character names
print('acts:', speeches['act'].nunique())
print('scenes:', len(speeches.groupby(['act', 'scene'])))
print('speeches:', len(speeches))
speeches['speaker'].value_counts()

acts: 5
scenes: 18
speeches: 926


speaker
TOBY              152
VIOLA             122
OLIVIA            119
FOOL              104
ANDREW             88
MALVOLIO           87
ORSINO             59
MARIA              59
FABIAN             51
SEBASTIAN          31
ANTONIO            27
CAPTAIN            10
FIRST OFFICER       5
CURIO               4
VALENTINE           3
SECOND OFFICER      3
SERVANT             1
PRIEST              1
Name: count, dtype: int64

Note: Feste is called FOOL in the Folger text, and Sir Toby and Sir Andrew are TOBY and ANDREW.

## Cleaning and lemmatizing

Same steps as 6-2: tokenize, POS tag, lowercase, keep only words, remove stopwords, lemmatize. NLTK's stopwords don't have old words like thou, thee, hath, so we added those.

In [7]:
stop_words = set(stopwords.words('english'))
stop_words.update(['thou', 'thee', 'thy', 'thine', 'ye', 'hath', 'hast', 'doth', 'dost',
                   'art', 'wilt', 'shalt', 'ay', 'nay', 'o', 'tis', 'shall', 'would', 'upon'])

lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def clean(text):
    words = []
    for word, tag in pos_tag(word_tokenize(text)):
        word = word.lower()
        if word.isalpha() and word not in stop_words:
            word = lemmatizer.lemmatize(word, get_wordnet_pos(tag))
            if word not in stop_words:
                words.append(word)
    return ' '.join(words)

print(clean('If music be the food of love, play on.'))

music food love play


In [8]:
speeches['clean_text'] = speeches['text'].apply(clean)
speeches.head(10)

,act,scene,speaker,text,clean_text
0,1,1,ORSINO,"If music be the food of love, play on. Give me...",music food love play give excess surfeit appet...
1,1,1,CURIO,"Will you go hunt, my lord?",go hunt lord
2,1,1,ORSINO,"What, Curio?",curio
3,1,1,CURIO,The hart.,hart
4,1,1,ORSINO,"Why, so I do, the noblest that I have. O, when...",noble mine eye see olivia first methought purg...
5,1,1,VALENTINE,"So please my lord, I might not be admitted, Bu...",please lord might admit handmaid return answer...
6,1,1,ORSINO,"O, she that hath a heart of that fine frame To...",heart fine frame pay debt love brother love ri...
7,1,2,VIOLA,"What country, friends, is this?",country friend
8,1,2,CAPTAIN,"This is Illyria, lady.",illyria lady
9,1,2,VIOLA,And what should I do in Illyria? My brother he...,illyria brother elysium perchance think sailor


## Save

In [9]:
# some short speeches (like "I will.") are empty after cleaning
speeches.to_csv('data/twelfth_night_speeches.csv', index=False)
directions_df.to_csv('data/twelfth_night_stage_directions.csv', index=False)